# Notebook 6: Energy Per Token Model
Based on Roofline Model (Williams et al., 2009)
Models energy consumption across quantization levels.

In [ ]:
!pip install matplotlib pandas numpy -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os

os.makedirs("results/tables", exist_ok=True)
os.makedirs("results/figures", exist_ok=True)

HBM_BW = 2e12
BOARD_POWER = 400
PEAK_FLOPS = 312e12
N_LAYERS, N_HEADS, HEAD_DIM = 24, 16, 64

seq_lengths = [128, 256, 512]
quant_configs = {"FP16": 2, "INT8": 1, "INT4": 0.5}
rows = []

for quant, db in quant_configs.items():
    for seq_len in seq_lengths:
        bytes_accessed = 2 * seq_len * N_LAYERS * N_HEADS * HEAD_DIM * db
        flops = 2 * seq_len * seq_len * N_HEADS * HEAD_DIM
        mem_time = bytes_accessed / HBM_BW
        compute_time = flops / PEAK_FLOPS
        energy_uj = round(max(mem_time, compute_time) * BOARD_POWER * 1e6, 4)
        rows.append({"quant": quant, "seq_len": seq_len, "energy_uj": energy_uj})
        print(f"{quant} seq={seq_len}: {energy_uj} uJ")

df = pd.DataFrame(rows)
df.to_csv("results/tables/energy_results.csv", index=False)
print("Saved energy_results.csv")

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(seq_lengths))
width = 0.25
for i, quant in enumerate(quant_configs.keys()):
    vals = df[df["quant"] == quant]["energy_uj"].values
    bars = ax.bar(x + i*width, vals, width, label=quant)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                f"{val:.1f}", ha='center', fontsize=8)
ax.set_title("Energy per Token (μJ) — Roofline Model (Williams et al., 2009)")
ax.set_xlabel("Sequence Length")
ax.set_ylabel("Energy (microjoules)")
ax.set_xticks(x + width)
ax.set_xticklabels(seq_lengths)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig("results/figures/energy_per_token.png", dpi=150)
print("Saved energy_per_token.png")
plt.show()